# Objects365 v2 val: boxes, exhaustive labels, classes matched to PROCTHOR30

[Objects365](https://www.objects365.org/) v2 val: web photos (indoor and outdoor), 365 classes, boxes only, every object
of every class annotated; crowd boxes carry `iscrowd`. Read with `habitat_embodied_sl/static_datasets/objects365.py`'s
`objects365_source`, i.e. detectron2's `load_coco_json` on `annotations/zhiyuan_objv2_val_fixname.json` (Detic's
`fix_o365_names.py` typo fixes), images in `val/images/v{1,2}/patch<N>/`. Needs `INSTALL.MD`'s Objects365 section run
first. Being exhaustive, Objects365 needs no ignored region: an unmatched detection of a kept class is a false
positive wherever it is.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("BASE_DIR", str(REPO_ROOT))

from common.env_utils.vocab_constants import CLASS_LABELS_PROCTHOR30
from common.vision.detic import matched_classes
from habitat_embodied_sl.static_datasets.detection_sources import showing_records
from habitat_embodied_sl.static_datasets.inspection import matched_table, plot_matched_counts, plot_verification, show_records

VOCAB, THRESHOLD = CLASS_LABELS_PROCTHOR30, 0.9

In [ ]:
from habitat_embodied_sl.static_datasets.objects365 import OBJECTS365_ROOT, objects365_source

source = objects365_source(OBJECTS365_ROOT / "annotations" / "zhiyuan_objv2_val_fixname.json", OBJECTS365_ROOT / "val")
records, objects365 = source.records, source.vocabulary
print(f"{len(records)} images, {sum(len(r['annotations']) for r in records)} boxes, {len(objects365.classes)} classes")
records[0]["file_name"], records[0]["annotations"][:2]

## A few examples

Random records, drawn with detectron2's `Visualizer` (masks where the source has them).

In [ ]:
show_records(records, objects365.classes)

## Classes matched to PROCTHOR30

`common/vision/detic.py`'s `matched_classes`: every Objects365 class whose CLIP text embedding is at least `THRESHOLD`
cosine-similar to some `VOCAB` word's, both prompted as `a <name>` -- the classes `objects365.py` starts from with
`--vocab` and `--sim-word-thresh`. Change `VOCAB`/`THRESHOLD` above to try another vocabulary.

In [ ]:
matches = matched_classes(objects365, VOCAB, THRESHOLD)
matched = {index for index, _, _ in matches}
matched_table(objects365, matches)

## Matched classes' frequency

Object count of each matched class over the images showing at least one (`detection_sources.py`'s `showing_records`,
every one here, as without `--max-images`). `objects365.py` keeps the classes with at least `MIN_OCCURRENCES` objects
(`vocabulary.py`, after merging spelling variants), in blue; the grey ones' objects are left out of its test set.

In [ ]:
showing = showing_records(records, matched, None)
plot_matched_counts(showing, objects365.classes, matched)

## Building the test set

```bash
PYTHONPATH=. python habitat_embodied_sl/static_datasets/objects365.py datasets/objects365_val --vocab <VOCAB words> --sim-word-thresh 0.9 --max-images 5000
```
writes `datasets/objects365_val/test/test.json` and its images in `datasets/objects365_val/images/`, registered as `objects365_val_test`
(`common/env_utils/dataset_registry.py`) for a training config's `DATASETS.TEST`.